# AI Assistive Room Setup Planner
**CSC 247 Semester Project**

A classical-AI constraint satisfaction program that arranges furniture around mobility, tactile, cognitive, and sensory needs.

## 1. What the project does
Enter a room and accessibility profile, or choose a preset. The planner searches for a labeled furniture layout.

## 2. Important scope note
One grid cell represents one foot. This is a classroom prototype—not ADA compliance, medical, safety, or architectural certification.

## 3. What makes it AI?
The program models furniture positions as variables, possible grid positions as domains, and accessibility rules as constraints. It searches alternatives, rejects conflicts, and backtracks. The improved solver then compares up to 50 valid layouts and selects the highest-scoring practical arrangement. Accessibility constraints remain mandatory: scoring only ranks layouts that already pass every enabled rule.

## 4. AI techniques
- Constraint satisfaction problem (CSP)
- Recursive backtracking
- Minimum Remaining Values (MRV)
- Forward checking
- Breadth-first search (BFS) for route validation

## 5. Planner implementation
The complete source is embedded below so this notebook needs no local project files.

In [ ]:
"""Core models for the AI Assistive Room Setup Planner.

One grid cell represents one foot. This is a classroom prototype, not an ADA
compliance or architectural certification tool.
"""

from dataclasses import dataclass, field
from collections import deque
from time import perf_counter
from typing import FrozenSet


Cell = tuple[int, int]


@dataclass(frozen=True)
class Furniture:
    name: str
    label: str
    width: int
    height: int
    color: str
    tags: FrozenSet[str] = frozenset()
    rotatable: bool = True


@dataclass(frozen=True)
class Placement:
    x: int
    y: int
    orientation: int = 0


@dataclass(frozen=True)
class AccessibilityProfile:
    name: str
    require_turning_space: bool = False
    require_essential_reach: bool = True
    require_tactile_reach: bool = False
    require_sensory_separation: bool = False
    sensory_distance: int = 3


@dataclass(frozen=True)
class RoomProblem:
    width: int
    height: int
    doors: tuple[Cell, ...]
    windows: tuple[Cell, ...] = ()
    furniture: tuple[str, ...] = ()
    profile: AccessibilityProfile = field(
        default_factory=lambda: AccessibilityProfile("standard")
    )


@dataclass(frozen=True)
class Preset:
    name: str
    description: str
    problem: RoomProblem


def oriented_size(item: Furniture, orientation: int) -> tuple[int, int]:
    """Return width and height after a 0° or 90° rotation."""
    if orientation not in (0, 90):
        raise ValueError("orientation must be 0 or 90 degrees")
    if orientation == 90:
        return item.height, item.width
    return item.width, item.height


FURNITURE_CATALOG = {
    "bed": Furniture("bed", "Bed", 2, 3, "#4C78A8", frozenset({"essential"})),
    "desk": Furniture("desk", "Desk", 2, 1, "#F58518", frozenset({"essential", "noisy"})),
    "chair": Furniture("chair", "Chair", 1, 1, "#54A24B", frozenset({"essential"}), False),
    "storage": Furniture("storage", "Storage", 1, 2, "#B279A2", frozenset({"essential"})),
    "tactile_station": Furniture("tactile_station", "Braille / Tactile Station", 1, 1, "#E45756", frozenset({"essential", "tactile"}), False),
    "quiet_corner": Furniture("quiet_corner", "Quiet Sensory Corner", 2, 2, "#72B7B2", frozenset({"essential", "quiet"}), False),
    "sensory_equipment": Furniture("sensory_equipment", "Sensory Equipment", 1, 1, "#FF9DA6", frozenset({"noisy"}), False),
}


MOBILITY = AccessibilityProfile("Wheelchair / limited mobility", require_turning_space=True)
SENSORY = AccessibilityProfile("Sensory-friendly", require_sensory_separation=True)
COMBINED = AccessibilityProfile(
    "Combined accessibility",
    require_turning_space=True,
    require_tactile_reach=True,
    require_sensory_separation=True,
)


PRESETS = {
    "mobility_bedroom": Preset(
        "Wheelchair-accessible bedroom",
        "A bedroom with a clear route and a 3×3-foot turning area.",
        RoomProblem(8, 7, ((3, 0),), ((7, 3),), ("bed", "desk", "chair", "storage"), MOBILITY),
    ),
    "sensory_room": Preset(
        "Sensory-friendly room",
        "Separates a quiet corner from noisy or stimulating equipment.",
        RoomProblem(8, 7, ((0, 3),), ((4, 6),), ("desk", "chair", "quiet_corner", "sensory_equipment"), SENSORY),
    ),
    "combined_room": Preset(
        "Combined-accessibility room",
        "Combines mobility, tactile, and sensory constraints.",
        RoomProblem(9, 8, ((4, 0),), ((8, 4),), ("bed", "desk", "chair", "storage", "tactile_station", "quiet_corner", "sensory_equipment"), COMBINED),
    ),
    "impossible_room": Preset(
        "Impossible-room test",
        "A deliberately crowded room that proves the solver can report no solution.",
        RoomProblem(3, 3, ((1, 0),), (), ("bed", "desk", "storage", "quiet_corner"), COMBINED),
    ),
}


def _on_edge(problem: RoomProblem, cell: Cell) -> bool:
    x, y = cell
    return (0 <= x < problem.width and 0 <= y < problem.height and
            (x in (0, problem.width - 1) or y in (0, problem.height - 1)))


def door_entry_cells(problem: RoomProblem) -> set[Cell]:
    """Return the interior cells immediately beside each edge door."""
    entries: set[Cell] = set()
    for x, y in problem.doors:
        candidates = []
        if x == 0:
            candidates.append((x + 1, y))
        if x == problem.width - 1:
            candidates.append((x - 1, y))
        if y == 0:
            candidates.append((x, y + 1))
        if y == problem.height - 1:
            candidates.append((x, y - 1))
        entries.update(c for c in candidates if 0 <= c[0] < problem.width and 0 <= c[1] < problem.height)
    return entries


def validate_problem(problem: RoomProblem) -> list[str]:
    """Return readable input errors; an empty list means input is valid."""
    errors: list[str] = []
    if problem.width <= 0 or problem.height <= 0:
        errors.append("Room width and height must be positive.")
        return errors
    if not problem.doors:
        errors.append("Add at least one door on a room edge.")
    for door in problem.doors:
        if not _on_edge(problem, door):
            errors.append(f"Door {door} must be a valid cell on a room edge.")
    for window in problem.windows:
        if not _on_edge(problem, window):
            errors.append(f"Window {window} must be a valid cell on a room edge.")
    if not problem.furniture:
        errors.append("Select at least one furniture item.")
    for name in problem.furniture:
        if name not in FURNITURE_CATALOG:
            errors.append(f"Unknown furniture item: {name}.")
            continue
        item = FURNITURE_CATALOG[name]
        fits = any(w <= problem.width and h <= problem.height for w, h in
                   {oriented_size(item, 0), oriented_size(item, 90) if item.rotatable else oriented_size(item, 0)})
        if not fits:
            errors.append(f"{item.label} is larger than the room.")
    return errors


def occupied_cells(item: Furniture, placement: Placement) -> set[Cell]:
    width, height = oriented_size(item, placement.orientation)
    return {(placement.x + dx, placement.y + dy)
            for dx in range(width) for dy in range(height)}


def placement_is_locally_valid(problem: RoomProblem, assignment: dict[str, Placement],
                               item_name: str, placement: Placement) -> bool:
    """Check boundaries, fixed features, doorway clearance, and overlap."""
    if item_name not in FURNITURE_CATALOG:
        return False
    cells = occupied_cells(FURNITURE_CATALOG[item_name], placement)
    if any(x < 0 or y < 0 or x >= problem.width or y >= problem.height for x, y in cells):
        return False
    forbidden = set(problem.doors) | set(problem.windows) | door_entry_cells(problem)
    if cells & forbidden:
        return False
    for other_name, other_placement in assignment.items():
        if other_name != item_name and cells & occupied_cells(FURNITURE_CATALOG[other_name], other_placement):
            return False
    return True


def generate_domain(problem: RoomProblem, item_name: str) -> list[Placement]:
    """Generate every in-bounds position and permitted orientation."""
    item = FURNITURE_CATALOG[item_name]
    orientations = (0, 90) if item.rotatable and item.width != item.height else (0,)
    values: list[Placement] = []
    for orientation in orientations:
        width, height = oriented_size(item, orientation)
        for y in range(problem.height - height + 1):
            for x in range(problem.width - width + 1):
                values.append(Placement(x, y, orientation))
    return values


def free_cells(problem: RoomProblem, assignment: dict[str, Placement]) -> set[Cell]:
    blocked = set(problem.windows)
    for name, placement in assignment.items():
        blocked |= occupied_cells(FURNITURE_CATALOG[name], placement)
    return {(x, y) for x in range(problem.width) for y in range(problem.height)} - blocked


def reachable_cells(problem: RoomProblem, assignment: dict[str, Placement]) -> set[Cell]:
    """Breadth-first search over free floor using four-direction movement."""
    free = free_cells(problem, assignment)
    starts = door_entry_cells(problem) & free
    reached = set(starts)
    queue = deque(starts)
    while queue:
        x, y = queue.popleft()
        for neighbor in ((x + 1, y), (x - 1, y), (x, y + 1), (x, y - 1)):
            if neighbor in free and neighbor not in reached:
                reached.add(neighbor)
                queue.append(neighbor)
    return reached


def _item_is_reachable(name: str, placement: Placement, reached: set[Cell]) -> bool:
    cells = occupied_cells(FURNITURE_CATALOG[name], placement)
    return any((x + dx, y + dy) in reached
               for x, y in cells for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)))


def _has_turning_square(problem: RoomProblem, assignment: dict[str, Placement]) -> bool:
    free = free_cells(problem, assignment)
    return any({(x + dx, y + dy) for dx in range(3) for dy in range(3)} <= free
               for x in range(problem.width - 2) for y in range(problem.height - 2))


def _sensory_separated(problem: RoomProblem, assignment: dict[str, Placement]) -> bool:
    quiet_cells: set[Cell] = set()
    noisy_cells: set[Cell] = set()
    for name, placement in assignment.items():
        tags = FURNITURE_CATALOG[name].tags
        if "quiet" in tags:
            quiet_cells |= occupied_cells(FURNITURE_CATALOG[name], placement)
        if "noisy" in tags:
            noisy_cells |= occupied_cells(FURNITURE_CATALOG[name], placement)
    if not quiet_cells or not noisy_cells:
        return True
    distance = min(abs(qx - nx) + abs(qy - ny)
                   for qx, qy in quiet_cells for nx, ny in noisy_cells)
    return distance >= problem.profile.sensory_distance


def evaluate_constraints(problem: RoomProblem, assignment: dict[str, Placement]) -> dict[str, bool]:
    """Evaluate enabled accessibility requirements for a complete layout."""
    reached = reachable_cells(problem, assignment)
    entries = door_entry_cells(problem)
    results = {
        "doorway_clear": bool(entries) and entries <= free_cells(problem, assignment),
        "connected_travel_path": bool(reached),
    }
    if problem.profile.require_essential_reach:
        essential = [(name, p) for name, p in assignment.items()
                     if "essential" in FURNITURE_CATALOG[name].tags]
        results["essential_items_reachable"] = all(
            _item_is_reachable(name, p, reached) for name, p in essential
        )
    if problem.profile.require_turning_space:
        results["clear_turning_space"] = _has_turning_square(problem, assignment)
    if problem.profile.require_tactile_reach:
        tactile = [(name, p) for name, p in assignment.items()
                   if "tactile" in FURNITURE_CATALOG[name].tags]
        results["tactile_station_reachable"] = bool(tactile) and all(
            _item_is_reachable(name, p, reached) for name, p in tactile
        )
    if problem.profile.require_sensory_separation:
        results["sensory_zones_separated"] = _sensory_separated(problem, assignment)
    return results


def validate_complete_assignment(problem: RoomProblem, assignment: dict[str, Placement]) -> tuple[bool, dict[str, bool]]:
    """Independently validate local placement and global accessibility rules."""
    complete = set(assignment) == set(problem.furniture)
    local = complete
    built: dict[str, Placement] = {}
    if complete:
        for name in problem.furniture:
            if not placement_is_locally_valid(problem, built, name, assignment[name]):
                local = False
                break
            built[name] = assignment[name]
    results = evaluate_constraints(problem, assignment) if complete else {}
    results = {"all_items_placed": complete, "no_overlap_or_fixed_conflict": local, **results}
    return all(results.values()), results


@dataclass(frozen=True)
class LayoutScore:
    """Explainable quality score for an already-valid room layout."""

    total: float
    breakdown: dict[str, float]


def score_layout(problem: RoomProblem, assignment: dict[str, Placement]) -> LayoutScore:
    """Score practical preferences without replacing accessibility constraints."""
    centers: list[tuple[float, float]] = []
    for name, placement in assignment.items():
        width, height = oriented_size(FURNITURE_CATALOG[name], placement.orientation)
        centers.append((placement.x + width / 2, placement.y + height / 2))

    if len(centers) < 2:
        furniture_spread = 0.0
    else:
        distances = [
            (abs(ax - bx) / problem.width + abs(ay - by) / problem.height) / 2
            for index, (ax, ay) in enumerate(centers)
            for bx, by in centers[index + 1:]
        ]
        furniture_spread = 40 * (sum(distances) / len(distances))

    wall_items = [name for name in assignment if name in {"bed", "storage"}]
    wall_hits = 0
    for name in wall_items:
        cells = occupied_cells(FURNITURE_CATALOG[name], assignment[name])
        if any(x in (0, problem.width - 1) or y in (0, problem.height - 1)
               for x, y in cells):
            wall_hits += 1
    wall_placement = 20 * wall_hits / len(wall_items) if wall_items else 0.0

    desk_near_window = 0.0
    if "desk" in assignment and problem.windows:
        item = FURNITURE_CATALOG["desk"]
        width, height = oriented_size(item, assignment["desk"].orientation)
        desk_center = (assignment["desk"].x + width / 2,
                       assignment["desk"].y + height / 2)
        distance = min(abs(desk_center[0] - x - .5) + abs(desk_center[1] - y - .5)
                       for x, y in problem.windows)
        max_distance = max(1.0, problem.width + problem.height - 2)
        desk_near_window = 20 * max(0.0, 1 - distance / max_distance)

    center_x = set(range(problem.width // 3, problem.width - problem.width // 3))
    center_y = set(range(problem.height // 3, problem.height - problem.height // 3))
    center_cells = {(x, y) for x in center_x for y in center_y}
    occupied: set[Cell] = set()
    for name, placement in assignment.items():
        occupied |= occupied_cells(FURNITURE_CATALOG[name], placement)
    blocked_ratio = len(occupied & center_cells) / len(center_cells) if center_cells else 0.0
    open_center = 20 * (1 - blocked_ratio)

    breakdown = {
        "furniture_spread": round(furniture_spread, 3),
        "wall_placement": round(wall_placement, 3),
        "desk_near_window": round(desk_near_window, 3),
        "open_center": round(open_center, 3),
    }
    return LayoutScore(round(sum(breakdown.values()), 3), breakdown)


@dataclass(frozen=True)
class SolverConfig:
    use_mrv: bool = False
    use_forward_checking: bool = False
    max_attempts: int = 200_000
    candidate_limit: int = 50


@dataclass
class SearchMetrics:
    attempts: int = 0
    backtracks: int = 0
    max_depth: int = 0
    elapsed_seconds: float = 0.0
    valid_candidates: int = 0


@dataclass
class SolveResult:
    status: str
    assignment: dict[str, Placement]
    constraints: dict[str, bool]
    metrics: SearchMetrics
    message: str = ""
    layout_score: LayoutScore | None = None


def solve(problem: RoomProblem, config: SolverConfig = SolverConfig()) -> SolveResult:
    """Solve the furniture CSP with backtracking and optional MRV/forward checking."""
    start = perf_counter()
    metrics = SearchMetrics()
    errors = validate_problem(problem)
    if errors:
        metrics.elapsed_seconds = perf_counter() - start
        return SolveResult("invalid_input", {}, {}, metrics, " ".join(errors))
    domains = {
        name: [p for p in generate_domain(problem, name)
               if placement_is_locally_valid(problem, {}, name, p)]
        for name in problem.furniture
    }
    assignment: dict[str, Placement] = {}
    hit_limit = False
    stop_search = False
    optimize = config.use_mrv and config.use_forward_checking
    best_answer: dict[str, Placement] | None = None
    best_score: LayoutScore | None = None

    def legal_values(name: str) -> list[Placement]:
        values = [p for p in domains[name]
                  if placement_is_locally_valid(problem, assignment, name, p)]
        if optimize:
            values.sort(key=lambda placement: (
                -score_layout(problem, {**assignment, name: placement}).total,
                placement.y,
                placement.x,
                placement.orientation,
            ))
        return values

    def search() -> dict[str, Placement] | None:
        nonlocal hit_limit, stop_search, best_answer, best_score
        metrics.max_depth = max(metrics.max_depth, len(assignment))
        if len(assignment) == len(problem.furniture):
            if not validate_complete_assignment(problem, assignment)[0]:
                return None
            candidate = dict(assignment)
            candidate_score = score_layout(problem, candidate)
            metrics.valid_candidates += 1
            if best_score is None or candidate_score.total > best_score.total:
                best_answer = candidate
                best_score = candidate_score
            if not optimize:
                return candidate
            if metrics.valid_candidates >= max(1, config.candidate_limit):
                stop_search = True
            return None
        remaining = [name for name in problem.furniture if name not in assignment]
        if config.use_mrv:
            choices = [(len(legal_values(name)), index, name)
                       for index, name in enumerate(remaining)]
            _, _, name = min(choices)
        else:
            name = remaining[0]
        for placement in legal_values(name):
            if metrics.attempts >= config.max_attempts:
                hit_limit = True
                return None
            metrics.attempts += 1
            assignment[name] = placement
            metrics.max_depth = max(metrics.max_depth, len(assignment))
            viable = True
            if config.use_forward_checking:
                viable = all(legal_values(other) for other in remaining if other != name)
            if viable:
                answer = search()
                if answer is not None:
                    return answer
                if hit_limit or stop_search:
                    assignment.pop(name, None)
                    return None
            assignment.pop(name, None)
        metrics.backtracks += 1
        return None

    answer = search()
    metrics.elapsed_seconds = perf_counter() - start
    if optimize and best_answer is not None:
        answer = best_answer
    if answer is not None:
        valid, constraints = validate_complete_assignment(problem, answer)
        assert valid
        selected_score = best_score or score_layout(problem, answer)
        if hit_limit:
            message = "Best valid layout found within the search budget."
        elif optimize:
            message = f"Best layout selected from {metrics.valid_candidates} valid candidates."
        else:
            message = "First valid layout found."
        return SolveResult("solved", answer, constraints, metrics, message, selected_score)
    if hit_limit:
        return SolveResult("limit_reached", {}, {}, metrics,
                           "Search limit reached; this does not prove that no layout exists.")
    return SolveResult("no_solution", {}, {}, metrics,
                       "The search exhausted every arrangement; no valid layout exists.")


def compare_solvers(problem: RoomProblem) -> tuple[SolveResult, SolveResult]:
    """Run basic search and MRV plus forward checking on the same room."""
    return (solve(problem, SolverConfig(False, False)),
            solve(problem, SolverConfig(True, True)))


CONSTRAINT_LABELS = {
    "all_items_placed": "All requested furniture is placed",
    "no_overlap_or_fixed_conflict": "Furniture does not overlap or block fixed features",
    "doorway_clear": "Doorway and interior entry space are clear",
    "connected_travel_path": "A connected four-direction travel path starts at the door",
    "essential_items_reachable": "Essential furniture is reachable from the travel path",
    "clear_turning_space": "A clear 3×3-foot wheelchair turning area is available",
    "tactile_station_reachable": "The Braille / tactile station is reachable",
    "sensory_zones_separated": "Quiet and noisy sensory zones are separated",
}


def constraint_messages(results: dict[str, bool]) -> list[str]:
    return [f"{'✓' if passed else '✗'} {CONSTRAINT_LABELS.get(name, name.replace('_', ' ').title())}"
            for name, passed in results.items()]


def metrics_table(basic: SolveResult, improved: SolveResult) -> list[dict[str, object]]:
    rows = []
    for label, result in (("Basic backtracking", basic), ("MRV + forward checking", improved)):
        rows.append({
            "solver": label,
            "status": result.status,
            "attempts": result.metrics.attempts,
            "backtracks": result.metrics.backtracks,
            "max_depth": result.metrics.max_depth,
            "elapsed_seconds": round(result.metrics.elapsed_seconds, 6),
        })
    return rows


def _first_turning_square(problem: RoomProblem, assignment: dict[str, Placement]) -> Cell | None:
    free = free_cells(problem, assignment)
    for y in range(problem.height - 2):
        for x in range(problem.width - 2):
            if {(x + dx, y + dy) for dx in range(3) for dy in range(3)} <= free:
                return x, y
    return None


def plot_layout(problem: RoomProblem, assignment: dict[str, Placement], ax=None):
    """Draw a labeled layout; returns the Matplotlib axis for reuse/testing."""
    import matplotlib.pyplot as plt
    from matplotlib.patches import Patch, Rectangle

    if ax is None:
        _, ax = plt.subplots(figsize=(8, 6))
    reached = reachable_cells(problem, assignment)
    if reached:
        xs, ys = zip(*reached)
        path = ax.scatter([x + .5 for x in xs], [y + .5 for y in ys], s=18,
                          color="#D9EAF7", marker="s", label="Reachable floor", zorder=0)
        path.set_gid("path")
    for x, y in problem.doors:
        patch = Rectangle((x, y), 1, 1, facecolor="#FFFFFF", edgecolor="#0072B2", linewidth=3)
        patch.set_gid("door")
        ax.add_patch(patch)
        ax.text(x + .5, y + .5, "DOOR", ha="center", va="center", fontsize=7, weight="bold")
    for x, y in problem.windows:
        patch = Rectangle((x, y), 1, 1, facecolor="#BDE3FF", edgecolor="#005A8D", hatch="//")
        patch.set_gid("window")
        ax.add_patch(patch)
    turning = _first_turning_square(problem, assignment) if problem.profile.require_turning_space else None
    if turning:
        patch = Rectangle(turning, 3, 3, facecolor="none", edgecolor="#009E73",
                          linewidth=2, linestyle="--")
        patch.set_gid("turning-area")
        ax.add_patch(patch)
        ax.text(turning[0] + 1.5, turning[1] + 1.5, "3×3 TURNING\nAREA",
                ha="center", va="center", color="#006B4E", fontsize=8, weight="bold")
    for name, placement in assignment.items():
        item = FURNITURE_CATALOG[name]
        width, height = oriented_size(item, placement.orientation)
        patch = Rectangle((placement.x, placement.y), width, height,
                          facecolor=item.color, edgecolor="#202124", linewidth=1.5,
                          alpha=.9, label=item.label)
        patch.set_gid("furniture")
        ax.add_patch(patch)
        ax.text(placement.x + width / 2, placement.y + height / 2, item.label,
                ha="center", va="center", fontsize=8, color="white", weight="bold", wrap=True)
    ax.set_xlim(0, problem.width)
    ax.set_ylim(0, problem.height)
    ax.set_aspect("equal")
    ax.set_xticks(range(problem.width + 1))
    ax.set_yticks(range(problem.height + 1))
    ax.grid(True, color="#C7C7C7", linewidth=.6)
    ax.set_xlabel("Width (feet; 1 cell = 1 foot)")
    ax.set_ylabel("Height (feet; 1 cell = 1 foot)")
    ax.set_title(f"Assistive Room Layout — {problem.profile.name}\nClassroom planning model; not ADA certification")
    handles = [Patch(facecolor="#FFFFFF", edgecolor="#0072B2", label="Door"),
               Patch(facecolor="#BDE3FF", edgecolor="#005A8D", hatch="//", label="Window")]
    handles += [Patch(facecolor=FURNITURE_CATALOG[n].color, label=FURNITURE_CATALOG[n].label)
                for n in assignment]
    ax.legend(handles=handles, bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
    return ax


## 6. Display helpers

In [ ]:
def show_result(problem, result):
    print(f"Status: {result.status}")
    print(result.message)
    print(f"Attempts: {result.metrics.attempts:,} | Backtracks: {result.metrics.backtracks:,} | Time: {result.metrics.elapsed_seconds:.4f}s")
    if result.status == "solved":
        for line in constraint_messages(result.constraints):
            print(line)
        if result.layout_score is not None:
            print(f"\nLayout quality score: {result.layout_score.total:.3f}/100")
            print(f"Valid layouts compared: {result.metrics.valid_candidates}")
            score_labels = {
                "furniture_spread": "Furniture spread",
                "wall_placement": "Bed/storage wall placement",
                "desk_near_window": "Desk near window",
                "open_center": "Open central space",
            }
            for key, label in score_labels.items():
                print(f"- {label}: {result.layout_score.breakdown[key]:.3f}")
            print("Scores rank already-valid layouts; they are not accessibility certification.")
        plot_layout(problem, result.assignment)
        import matplotlib.pyplot as plt
        plt.show()

def run_preset(name="mobility_bedroom", improved=True, max_attempts=200000):
    """Solve and display one of the four built-in examples."""
    if name not in PRESETS:
        print("Unknown preset. Choose:", ", ".join(PRESETS))
        return None
    preset = PRESETS[name]
    print(preset.name, "—", preset.description)
    result = solve(preset.problem, SolverConfig(improved, improved, max_attempts))
    show_result(preset.problem, result)
    return result

## 7. Preset chooser
Change the dropdown in Colab, then run the cell.

In [ ]:
# @title Choose a preset
preset_name = "mobility_bedroom" # @param ["mobility_bedroom", "sensory_room", "combined_room", "impossible_room"]
use_improved_solver = True # @param {type:"boolean"}
run_preset(preset_name, use_improved_solver)

## 8. Compare the AI techniques
Basic backtracking stops at its first valid solution. The improved solver uses MRV and forward checking, compares up to 50 valid layouts, and selects the highest quality score. The four contributions reward furniture spread, bed/storage wall placement, a desk near a window, and open central space. Accessibility constraints remain mandatory for every candidate.

In [ ]:
basic, improved = compare_solvers(PRESETS[preset_name].problem)
for row in metrics_table(basic, improved):
    print(row)

## 9. Custom-room function
Coordinates use `x,y`, beginning at `0,0` in the lower-left. Doors and windows must be on an edge.

In [ ]:
def _parse_cells(text, field_name):
    cells = []
    if not text.strip():
        return tuple(cells)
    for part in text.split(";"):
        pieces = part.strip().split(",")
        if len(pieces) != 2:
            raise ValueError(f"{field_name} must use x,y; separate multiple coordinates with semicolons.")
        try:
            cells.append((int(pieces[0]), int(pieces[1])))
        except ValueError as exc:
            raise ValueError(f"{field_name} contains a non-integer coordinate.") from exc
    return tuple(cells)

def run_custom_room(width, height, door_coordinates, window_coordinates,
                    furniture_names, profile_name="combined", improved=True,
                    max_attempts=200000):
    """Validate, solve, and display a custom room."""
    try:
        doors = _parse_cells(door_coordinates, "Door coordinates")
        windows = _parse_cells(window_coordinates, "Window coordinates")
        furniture = tuple(x.strip() for x in furniture_names.split(",") if x.strip())
        if not furniture:
            raise ValueError("Select at least one furniture item.")
        profiles = {"standard": AccessibilityProfile("Standard"),
                    "mobility": MOBILITY, "sensory": SENSORY, "combined": COMBINED}
        if profile_name not in profiles:
            raise ValueError("Profile must be standard, mobility, sensory, or combined.")
        problem = RoomProblem(int(width), int(height), doors, windows, furniture, profiles[profile_name])
        errors = validate_problem(problem)
        if errors:
            print("Invalid input:")
            for error in errors:
                print("-", error)
            return None
        result = solve(problem, SolverConfig(improved, improved, int(max_attempts)))
        show_result(problem, result)
        return result
    except (TypeError, ValueError) as error:
        print("Invalid input:", error)
        return None

## 10. Simple custom form
Edit the fields and run this cell. Available furniture: `bed`, `desk`, `chair`, `storage`, `tactile_station`, `quiet_corner`, `sensory_equipment`.

In [ ]:
# @title Design your own room
room_width = 8 # @param {type:"integer"}
room_height = 7 # @param {type:"integer"}
door_coordinates = "3,0" # @param {type:"string"}
window_coordinates = "7,3" # @param {type:"string"}
furniture_names = "bed,desk,chair,storage,tactile_station" # @param {type:"string"}
accessibility_profile = "mobility" # @param ["standard", "mobility", "sensory", "combined"]
improved_search = True # @param {type:"boolean"}
maximum_attempts = 200000 # @param {type:"integer"}
run_custom_room(room_width, room_height, door_coordinates, window_coordinates,
                furniture_names, accessibility_profile, improved_search, maximum_attempts)

## 11. How to interpret the result
`solved` means every enabled accessibility check passed. The layout quality score then ranks only those valid arrangements; it is a classroom preference score, not certification. `no_solution` means all arrangements were exhausted. `limit_reached` means the cap stopped search before any valid layout was found and does **not** prove impossibility.

## 12. Suggested class demo
1. Run the mobility preset.
2. Point out the clear door, reachable floor, and turning square.
3. Compare solver metrics.
4. Run the impossible preset and explain its status.

## 13. Limitations and future work
The one-foot grid is simplified. A future version could use finer measurements, multiple doors, personalized reach ranges, and professional accessibility data.